# DNA nanostar sequence-to-rheology workflow: step-by-step tutorial

This notebook walks through the key steps of the multiscale workflow, from
nucleotide-resolution oxDNA simulations to Bayesian optimisation of the bead-spring model, to graph-based rheology. It is
meant as a compact, runnable overview — each section only shows the essential steps and points to the
full reference notebooks (in this same repository) for the complete analysis, additional angles, and
production-scale runs.

**Folder layout assumed by this notebook**

All file loading and saving below uses paths relative to this notebook's folder, so the notebook and its
input/output files can be run together from one folder:

```
./
├── tutorial_workflow.ipynb   (this notebook)
├── input_oxDNA.txt           (example oxDNA input script)
├── data/                     (oxDNA trajectories, target angle files)
└── sims/                     (ReaDDy simulation output)
```

Adjust the paths below if you keep data elsewhere. The custom helper modules imported below
(`NCG_target_angle_eval_fcts`, `NCG_readdy_sim_4`, `NCG_Bay_opt_readdy_sim_3`, `NCG_readdy_sim_eval_2`,
`NCG_readdy_sim_eval_3`) are provided alongside the full reference notebooks in this repository and are
not redefined here.

## 1. Setting up an oxDNA simulation

[oxDNA](https://dna.physics.ox.ac.uk/) is a nucleotide-resolution coarse-grained model of DNA (and RNA)
that represents each nucleotide as a rigid body with base-pairing, stacking, and backbone interactions.
We use it to obtain reference conformational statistics (angle distributions) for a single DNA nanostar,
which later serve as optimisation targets for the coarse-grained bead-spring model in ReaDDy.

An oxDNA simulation is run from the command line and needs three inputs: an **input file** (simulation
parameters), a **topology file** (strand/base connectivity), and an **initial configuration file**
(starting 3D coordinates). The example input script `input_oxDNA.txt` in this folder sets up an MD run of
the four-armed nanostar at 20°C and 150 mM NaCl for 5e7 steps, and defines several output observables
(potential energy, particle positions, and hydrogen-bond counts per arm).

In [ ]:
# The oxDNA input file used for the nanostar simulations in this project is provided as
# input_oxDNA.txt. Key options to note:
#   interaction_type = DNA2      -> sequence-dependent oxDNA2 model
#   T = 20C, salt_concentration = 0.15   -> matches the experimental reference conditions
#   steps = 5e7                  -> production run length
#   topology / conf_file         -> nanostar topology and starting configuration (not included here,
#                                    see reference notebook / repository data folder)
with open("input_oxDNA") as f:
    print(f.read())


In [ ]:
# oxDNA itself is a compiled C++/CUDA program and is run from the command line (or from a notebook
# via the `!` shell escape), not as a Python import. This requires a working oxDNA installation and,
# for the CUDA backend used here, a GPU.

# Example (uncomment to run; requires oxDNA to be installed and on your PATH):
# !oxDNA input_oxDNA.txt


## 2. Evaluating angles from an oxDNA trajectory

Once the oxDNA simulation has finished, we extract angle distributions that characterise nanostar
flexibility directly from the trajectory. Below we show this for a single angle, the **arm axis angle**
(the angle within one nanostar arm, capturing double-strand bending flexibility); the full reference
notebook (`NCG_target_angles_x_motif`) computes five further angles (base, opposing-arm, sticky-end,
hairpin sticky-end, and fused-arm angles) in the same way.

The resulting target distribution is saved to disk so it can be used as an optimisation target for the
bead-spring model in the next step.

In [ ]:
from NCG_target_angle_eval_fcts import read_positions_2, conf_angles_3

# Read the oxDNA trajectory (nucleotide positions per frame)
read_sim_coords = read_positions_2(
    file_name="data/trajectory_1.dat",
    num_nucl=196,
    time_start=0,
    time_stop=50000,
    time_step=1,
)

# Indices of nucleotides defining the arm-axis vectors for each of the four arms
# (see NCG_target_angles_x_motif for how these were determined from the nanostar topology)
arm_vector_indices = [  [  [*np.arange(105,115),*np.arange(186,196)], [*np.arange(109,119),*np.arange(182,192)] ,[*np.arange(109,119),*np.arange(182,192)] ,[*np.arange(178,188),*np.arange(113,123)] ]
                      , [  [*np.arange(154,164),*np.arange(39,49)]   ,[*np.arange(158,168),*np.arange(35,45)], [*np.arange(158,168),*np.arange(35,45)] ,[*np.arange(31,41),*np.arange(162,172)]  ]
                      , [  [*np.arange(7,17),*np.arange(88,98)]   ,[*np.arange(11,21),*np.arange(84,94)] ,[*np.arange(11,21),*np.arange(84,94)] ,[*np.arange(80,90),*np.arange(15,25)]  ]
                      , [  [*np.arange(56,66),*np.arange(137,147)]   ,[*np.arange(60,70),*np.arange(133,143)] ,[*np.arange(60,70),*np.arange(133,143)] ,[*np.arange(129,139),*np.arange(64,74)]  ]
                     ]

# Compute the arm axis angle for every frame and every arm, discard the initial equilibration frames
angles_arm_axis = conf_angles_3(pos=read_sim_coords[:], arm_vector_indices=arm_vector_indices)[300:]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

values_angle_arm_axis_target = angles_arm_axis.flatten()

plt.figure(figsize=(3, 3))
plt.hist(values_angle_arm_axis_target * 180 / np.pi, bins=np.arange(0, 190, 1), density=True)
plt.xlabel("Arm axis angle [deg]")
plt.ylabel("Normalized frequency")
plt.grid()
plt.show()

# Save the target angle distribution, to be used for Bayesian optimisation of the bead-spring model
#np.save("data/values_angle_arm_axis_target.npy", values_angle_arm_axis_target)


## 3. Setting up a ReaDDy simulation of the bead-spring model

[ReaDDy](https://readdy.github.io/) is a particle-based reaction-diffusion simulation platform that we
use to run a coarse-grained bead-spring model of the DNA nanostar. Each nanostar arm is represented by a
short chain of beads (roughly 8 base pairs per bead), connected by harmonic bond and angle potentials;
sticky ends at the arm tips can reversibly bind other sticky ends via a stochastic fusion/splitting
reaction, allowing whole networks of nanostars to assemble.

Simulations are launched through a helper function `run_sim_3`, which takes a dictionary of physical and
simulation parameters and writes the resulting trajectory to an HDF5 file. `run_p_dict` runs several such
simulations in parallel (e.g. independent repeats). The example below sets up the physical parameters
(diffusion constant, bond/angle spring constants, electrostatics, reaction rates) for a single nanostar,
more details can be found in `NCG_BO_opt_x_motif` and `NCG_run_networking_sim_1`.

In [ ]:
from NCG_readdy_sim_4 import (
    Debye_length, Debye_Hueckel_pot_rep_strength, Diff_ES, visc_T, run_sim_3, tcl_file_gen_3, all_types,
)
from NCG_Bay_opt_readdy_sim_3 import run_p_dict

# Units and system temperature
custom_units = {"length_unit": "nanometer", "time_unit": "nanoseconds", "energy_unit": "joule"}
system_temperature = 273.15 + 20  # 20 degC, matching oxDNA reference conditions

# Diffusion constant of an 8-bp coarse-grained bead (Einstein-Smoluchowski, sphere approximation)
visc_T_sys = visc_T(system_temperature)
Diff_sys_temp = Diff_ES(T=system_temperature, r=2.72 / 2 * 1e-9, eta=visc_T_sys) * 1e9  # nm^2/ns




In [ ]:
###################
#fixed params:
###################

#bond potential single/double stranded DNA
k_bond_ds=2*10**-19
k_bond_ss=1*10**-19

#bond eq. distance
eq_dist_bb=2.72 

#bond eq. distance fused sticky-ends
eq_dist_fused_bb=2/3 *2.72
eq_dist_bb_s=2/3 *2.72 # distance arm-stickyend

#bond potential surf
k_angle_ss_surf=2e-20  #2.8666666666666663e-20     #2*10**-21
k_angle_ds_surf=3.6666666666666664e-20     #2*10**-20

#angle between monomers V=k*(theta-theta_0)^2
theta_0=np.pi
theta_90=np.pi/2

theta_60=np.pi *1/3
theta_120=np.pi * 2/3
theta_270=np.pi * 3/2
theta_base_y=np.pi*2/3

#volume exclusion
#har rep
k_ex_ds_ds=2*10**-20  *5
k_ex_ss_ds=8*10**-21  *5
k_ex_ss_ss=4*10**-21  *5
dist_ex=1.5          *1 

#DH
c_NaCl = 150 # NaCl concentration in milli mol/L
L_1=6.022*10**23 *c_NaCl * (1.602*10**-19)**2 

charge_eff=0.815 #https://pubs.aip.org/aip/jcp/article/142/23/234901/193554/Introducing-improved-structural-properties-and
DH_len=Debye_length(L=L_1,T=system_temperature) *10**9 #for nm
DH_strength_ds_ds=Debye_Hueckel_pot_rep_strength(q1=charge_eff*16,q2=charge_eff*16,T=system_temperature)*10**9 #for nm
DH_strength_ss_ds=Debye_Hueckel_pot_rep_strength(q1=charge_eff*8,q2=charge_eff*16,T=system_temperature)*10**9 #for nm
DH_strength_ss_ss=Debye_Hueckel_pot_rep_strength(q1=charge_eff*8,q2=charge_eff*8,T=system_temperature)*10**9 #for nm
DH_cutoff=DH_len*4

volume_excl_type="DH"

print("Debye length at",c_NaCl,"mM NaCl:", DH_len, "nm", "cutoff at", DH_cutoff, "nm")
print("DH pot strength ds-ds:", DH_strength_ds_ds, "J*nm")
print("DH pot strength ss-ds:", DH_strength_ss_ds, "J*nm")
print("DH pot strength ss-ss:", DH_strength_ss_ss, "J*nm")
print("Energy of rep. ds-ds at 1 nm:",DH_strength_ds_ds/1 *np.exp(-1/DH_len),"J" )
#DH_strength_ds_ds=DH_strength_ds_ds
#DH_strength_ss_ds=DH_strength_ss_ds
#DH_strength_ss_ss=DH_strength_ss_ss
#DH_cutoff=DH_cutoff

#probabilities bound/unbound
p_SE_fused_x=99.8
p_SE_fused_ortho_x=99.8

p_SE_HP_x=83.8
p_SE_HP_ortho_x=59.9

p_surf_HP_x=6.6
p_surf_HP_ortho_x=6.6

#binding/unbinding rates
#rate of selecting fission reaction in sim
k_fiss_rate_sel=10**7
#y motif
k_b_y_1=0.05
k_ub_y_1=0.001

#y motif hp
k_b_y_hp_1=0
k_ub_y_hp_1=0

#############

#x motif 1
k_b_x_1=0     #0.05
k_ub_x_1=0    #0.001

#x motif 1 surface
k_b_x_surf_1=k_b_x_1 
k_ub_x_surf_1=k_ub_x_1

#x motif 1 hp
k_b_x_hp_1=0      #0.005
k_ub_x_hp_1=0      #k_b_x_hp_1 * (100-p_SE_HP_x)/p_SE_HP_x 

#x motif 1 surf hp
k_b_x_surf_hp_1=0     #0.005
k_ub_x_surf_hp_1=0     #k_b_x_surf_hp_1 * (100-p_surf_HP_x)/p_surf_HP_x

############

#x motif 2
k_b_x_2=0
k_ub_x_2=0

#x motif 2 surface
k_b_x_surf_2=0 
k_ub_x_surf_2=0

#x motif 2 hp
k_b_x_hp_2=0
k_ub_x_hp_2=0

#x motif 2 surf hp
k_b_x_surf_hp_2=0
k_ub_x_surf_hp_2=0

############

#x motif conrad

#x motif 1
k_b_x_c1=0    #0.05
k_ub_x_c1=0    #0.001

#x motif 1 hp
k_b_x_hp_c1=0      #0.005
k_ub_x_hp_c1=0      #k_b_x_hp_1 * (100-p_SE_HP_x)/p_SE_HP_x 


#time step
t_step=10*10**-3  #standard: 10*10**-3 #appears to work with up to 25*10**-3

#reaction scale
rea_s=2.7
rea_c1_s=2.7



#general simulation parameters
system_temperature=system_temperature
box_size=[150,150,150]
custom_units=custom_units
periodic_bc_x=False
periodic_bc_y=False
periodic_bc_z=False

r_sp=15 #radius of spherical inclusion pot
sp_incl_orig=[0,0,0] #origin of sperical inclusion pot
k_sp=0 #k of spherical inclusion, if zero no spherical inclusion
box_a=[-60,-60,-60] #offset of box potential from centre
box_b=[120,120,120] #total length of box potential
k_box=1 *10**-20 #k of box potential


#number of particles
num_y_motif_1=0
num_x_motif_1=0
num_x_motif_2=0
num_x_motif_c1=1
num_surface_1=0

#surface section lengths
num_rep_unit=1
len_fill_surf_a=2
len_binding_surf=4
len_fill_surf_b=2


#option to use HP template
use_y_1_HP_template=False
use_x_1_HP_template=False
use_x_2_HP_template=False
use_x_c1_HP_template=False

#positions of particles
y_motif_1_ini=[np.asarray([[0,0,0]])]
y_motif_1_rot=False

x_motif_1_ini=[np.asarray([[0,0,0]])]
x_motif_1_rot=False

x_motif_2_ini=[np.asarray([[-10,0,0]])]
x_motif_2_rot=False

x_motif_c1_ini=[np.asarray([[0,0,0]])] #[np.asarray([[0,0,0]])]
x_motif_c1_rot=False

surf_1_ini=[np.asarray([[-40,-40,-40]])]

#time steps
t_tot=500000  #500000

#evaluate reactions


#folder
folder="sims/single_motif/"
name="tutorial_sim_"
add_num_1=0
     

#other options

sim_kernel="SingleCPU"
sim_threads=1
track_num_particles=["y_link_a_1","y_link_fused_a_1","y_link_b_1","y_link_fused_b_1","y_link_c_1","y_link_fused_c_1",
                                                                   "x_link_a_1","x_link_fused_a_1","x_link_b_1","x_link_fused_b_1","x_link_c_1","x_link_fused_c_1","x_surf_link_d_1","x_surf_link_fused_d_1",
                                                                    "x_link_a_2","x_link_fused_a_2","x_link_b_2","x_link_fused_b_2","x_link_c_2","x_link_fused_c_2","x_surf_link_d_2","x_surf_link_fused_d_2",
                                                                    'x_link_hp_a_1', 'x_link_hp_b_1', 'x_link_hp_c_1',"x_surf_link_hp_d_1",
                                                                    'x_link_hp_a_2', 'x_link_hp_b_2', 'x_link_hp_c_2',"x_surf_link_hp_d_2",
                                                                    "x_link_a_c1","x_link_fused_a_c1","x_link_b_c1","x_link_fused_b_c1","x_link_c_c1","x_link_fused_c_c1","x_link_d_c1","x_link_fused_d_c1",
                                                                    'x_link_hp_a_c1', 'x_link_hp_b_c1', 'x_link_hp_c_c1',"x_link_hp_d_c1",
                                                                   ]
eval_reac=True
eval_top_reac=True

checkpoint_path=None #"ML_parameter_opt_1/NCG_test_1/"

new_checkpoint=None
max_n_saves=1 #max checkpoint saves
rec_stride=100 #recording stride
sim_skin=1000 #simulation skin


# Angle-potential spring constants and equilibrium angles are the tunable parameters optimised in
# Section 4 below; here we use example (unoptimised) values in the middle of their allowed range.
#wide param range
k_angle_ds_base_range=np.array([1*10**-24,1*10**-22,1*10**-20,1*10**-18])   #np.linspace(1*10**-24  ,1*10**-18 ,10) 
k_angle_ds_arm_range=np.array([1*10**-24,1*10**-22,1*10**-20,1*10**-18])   #np.linspace(1*10**-24  ,1*10**-18 ,10)
k_angle_ds_oppa_range=np.array([1*10**-24,1*10**-22,1*10**-20,1*10**-18])   #np.linspace(1*10**-24  ,1*10**-18 ,10)
k_angle_ss_link_range=np.array([1*10**-24,1*10**-22,1*10**-20,1*10**-18])   #np.linspace(1*10**-24  ,1*10**-18 ,10)
k_angle_ds_fused_link_range=np.array([1*10**-24,1*10**-22,1*10**-20,1*10**-18])   #np.linspace(1*10**-24  ,1*10**-18 ,10)
k_angle_ds_hp_range=np.array([1*10**-24,1*10**-22,1*10**-20,1*10**-18])   #np.linspace(1*10**-24  ,1*10**-18 ,10)
theta_base_x_range=np.linspace(75 ,120 ,4)*np.pi/180 #np.linspace(40 ,120 ,10)
theta_oppa_x_range=np.linspace(135 ,180 ,4)*np.pi/180 #np.linspace(140 ,180 ,10)

#Prior to optimization, use parameter value from middle of range for unoptimized sim
k_angle_ds_base=k_angle_ds_base_range[2]
k_angle_ds_arm=k_angle_ds_arm_range[2]
k_angle_ds_oppa=k_angle_ds_oppa_range[2]
k_angle_ss_link=k_angle_ss_link_range[2]
k_angle_ds_fused_link=k_angle_ds_fused_link_range[2]
k_angle_ds_hp=k_angle_ds_hp_range[2]
theta_base_x=theta_base_x_range[2]
theta_oppa_x=theta_oppa_x_range[2]


#input dictionary
run_sim_input={"name":name,
             "folder":folder,
             "add_num_1":add_num_1,
              
             "system_temperature":system_temperature,
             "box_size":box_size,
             "custom_units":custom_units,
             "periodic_bc_x":periodic_bc_x,
             "periodic_bc_y":periodic_bc_y,
             "periodic_bc_z":periodic_bc_z,
              
             "r_sp":r_sp,
             "sp_incl_orig":sp_incl_orig,
             "k_sp":k_sp,
             "box_a":box_a,
             "box_b":box_b,
             "k_box":k_box,
    
              
             "num_y_motif_1":num_y_motif_1,
             "num_x_motif_1":num_x_motif_1,
             "num_x_motif_2":num_x_motif_2,
             "num_x_motif_c1":num_x_motif_c1,
             "num_surface_1":num_surface_1,
             "num_rep_unit":num_rep_unit,
             "len_fill_surf_a":len_fill_surf_a,
             "len_binding_surf":len_binding_surf,
             "len_fill_surf_b":len_fill_surf_b,
             "y_motif_1_ini":y_motif_1_ini,
             "x_motif_1_ini":x_motif_1_ini,
             "x_motif_2_ini":x_motif_2_ini,
             "x_motif_c1_ini":x_motif_c1_ini,
             "surf_1_ini":surf_1_ini,
             "y_motif_1_rot":y_motif_1_rot,
             "x_motif_1_rot":x_motif_1_rot,
             "x_motif_2_rot":x_motif_2_rot,
             "x_motif_c1_rot":x_motif_c1_rot,
                 
             "use_y_1_HP_template":use_y_1_HP_template,
             "use_x_1_HP_template":use_x_1_HP_template,
             "use_x_2_HP_template":use_x_2_HP_template,
             "use_x_c1_HP_template":use_x_c1_HP_template,
                 
             "Diff_sys_temp":Diff_sys_temp,
             "k_bond_ds":k_bond_ds,
             "k_bond_ss":k_bond_ss,
             "eq_dist_bb":eq_dist_bb,
             "eq_dist_fused_bb":eq_dist_fused_bb,
             "eq_dist_bb_s":eq_dist_bb_s,
             "k_angle_ds_base":k_angle_ds_base,
             "k_angle_ds_arm":k_angle_ds_arm,
             "k_angle_ds_oppa":k_angle_ds_oppa,
             "k_angle_ss_link":k_angle_ss_link,
             "k_angle_ds_fused_link":k_angle_ds_fused_link,
             "k_angle_ds_hp":k_angle_ds_hp,
             "k_angle_ss_surf":k_angle_ss_surf,
             "k_angle_ds_surf":k_angle_ds_surf,
             "theta_0":theta_0,
             "theta_90":theta_90,
             "theta_60":theta_60,
             "theta_120":theta_120,
             "theta_270":theta_270,
             "theta_base_x":theta_base_x,
             "theta_base_y":theta_base_y,
             "theta_oppa_x":theta_oppa_x,
             
             "volume_excl_type":volume_excl_type,
             "k_ex_ds_ds":k_ex_ds_ds,
             "k_ex_ss_ds":k_ex_ss_ds,
             "k_ex_ss_ss":k_ex_ss_ss,
             "dist_ex":dist_ex,
             "DH_strength_ds_ds":DH_strength_ds_ds,
             "DH_strength_ss_ds":DH_strength_ss_ds,
             "DH_strength_ss_ss":DH_strength_ss_ss,
             "DH_len":DH_len,
             "DH_cutoff":DH_cutoff,
              
              
             "k_fiss_rate_sel":k_fiss_rate_sel,
             "k_b_y_1":k_b_y_1,
             "k_ub_y_1":k_ub_y_1,
             "k_b_y_hp_1":k_b_y_hp_1,
             "k_ub_y_hp_1":k_ub_y_hp_1,
             "k_b_x_1":k_b_x_1,
             "k_ub_x_1":k_ub_x_1,
             "k_b_x_surf_1":k_b_x_surf_1,
             "k_ub_x_surf_1":k_ub_x_surf_1,
             "k_b_x_hp_1":k_b_x_hp_1,
             "k_ub_x_hp_1":k_ub_x_hp_1,
             "k_b_x_surf_hp_1":k_b_x_surf_hp_1,
             "k_ub_x_surf_hp_1":k_ub_x_surf_hp_1,
             "k_b_x_2":k_b_x_2,
             "k_ub_x_2":k_ub_x_2,
             "k_b_x_surf_2":k_b_x_surf_2,
             "k_ub_x_surf_2":k_ub_x_surf_2,
             "k_b_x_hp_2":k_b_x_hp_2,
             "k_ub_x_hp_2":k_ub_x_hp_2,
             "k_b_x_surf_hp_2":k_b_x_surf_hp_2,
             "k_ub_x_surf_hp_2":k_ub_x_surf_hp_2,
             "k_b_x_c1":k_b_x_c1,
             "k_ub_x_c1":k_ub_x_c1,
             "k_b_x_hp_c1":k_b_x_hp_c1,
             "k_ub_x_hp_c1":k_ub_x_hp_c1,
                 
             "t_step":t_step,
             "rea_s":rea_s,
             "rea_c1_s":rea_c1_s,
             "t_tot":t_tot,
             
             "sim_kernel":sim_kernel,
             "sim_threads":sim_threads,
             "track_num_particles":track_num_particles,
             "eval_reac":eval_reac,
             "eval_top_reac":eval_top_reac,
             "checkpoint_path":checkpoint_path,
             "new_checkpoint":new_checkpoint,
             "max_n_saves":max_n_saves,
             "rec_stride":rec_stride,
             "sim_skin":sim_skin
    
    
}

# run_p_dict(fct_list, params_list) launches one or more simulations, optionally in parallel
#run_p_dict([run_sim_3], [run_sim_input])


## 4. Bayesian optimisation of one bead-spring angle (arm axis angle)

The bead-spring model's angle-potential parameters are tuned with Bayesian optimisation (BoTorch) so that
the resulting angle distributions match the oxDNA targets from Section 2. The full optimisation is split
into several low-dimensional sub-problems (per angle / group of correlated angles); here we show the
simplest case, a 1D optimisation of the arm-axis spring constant `k_angle_ds_arm` against the arm-axis
angle target saved above. The complete, 8-parameter split-optimisation strategy is implemented in
`NCG_BO_opt_x_motif`.

Each optimisation round: (1) runs a batch of simulations with candidate parameter values, (2) evaluates
the resulting angle distribution against the target with a fitness function, (3) fits a Gaussian process
to all (parameter, fitness) pairs observed so far, and (4) proposes the next candidate parameter value by
maximising an acquisition function.

## Supporting code for BO

Here we use the Botorch implementation to run the actual BO step. Using the  custom scaler functions below gave better stability than using scalers from other pacakges.

In [ ]:
#use botorch implementation 
#initialize GP model, train on input and predict next params
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
from botorch.models import SingleTaskGP
from botorch.models.transforms import Normalize, Standardize
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood
from gpytorch.likelihoods import FixedNoiseGaussianLikelihood
from botorch.acquisition import LogExpectedImprovement
from botorch.acquisition import ExpectedImprovement
from botorch.acquisition import UpperConfidenceBound
from botorch.optim import optimize_acqf
from NCG_Bay_opt_readdy_sim_3 import eval_sim_c1_1_aa, run_gp_11
import pandas as pd
from sklearn.preprocessing import FunctionTransformer

def apply_gp_bo_1(x_rs,x_train_rs,x,f_train_rs,seed,dim_x,dim_y,num_restarts,raw_samples,scaler_x):
    #x_rs: full rescaled parameter space  -> no actual rescaling needed with internal rescaling by botorch
    #x_train_rs: rescaled training values -> no actual rescaling needed with internal rescaling by botorch
    #x: full non-rescaled parameter space -> only needed to get bounds
    #initialize and train GP model

    #set random seed for torch and numpy, each round gets different seed based on number of rounds
    torch.manual_seed(seed+len(f_train_rs))
    np.random.seed(seed+len(f_train_rs))

    #fit scaler on full range of values
    scaler_x1=scaler_x()
    scaler_x1.fit(x) 
    x=scaler_x1.transform(x)
    #print(x)
    #minimum and maximum values for bounds
    min_bounds = np.min(x, axis=0)
    max_bounds = np.max(x, axis=0)
    bounds = np.vstack((min_bounds, max_bounds))
    #print(bounds)
    bounds = torch.from_numpy(bounds)
    #print(x_train_rs,f_train_rs)

    #x and f as arrays
    x_train_rs=np.array(x_train_rs)
    f_train_rs=np.array(f_train_rs).reshape(-1,1)

    #option to scale input before botorch 
    x_train_rs=scaler_x1.transform(x_train_rs)
    #print(x_train_rs,f_train_rs)

    #x and f as torch tensor
    x_train_rs=torch.from_numpy(x_train_rs)
    f_train_rs=torch.from_numpy(f_train_rs)
    
    #print(x_train_rs.shape,f_train_rs.shape)
    #option to specify noise
    #train_fvar = torch.zeros_like(f_train_rs)  # Zero noise
    #likelihood = FixedNoiseGaussianLikelihood(noise=train_fvar)
    #gp=SingleTaskGP(train_X=x_train_rs,train_Y=f_train_rs,**params_SingleTaskGP_dict)
    gp=SingleTaskGP(train_X=x_train_rs,train_Y=f_train_rs,input_transform=Normalize(d=dim_x),outcome_transform=Standardize(m=dim_y))
    

    mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
    fit_gpytorch_mll(mll)

    logEI = LogExpectedImprovement(model=gp, best_f=f_train_rs.max())
    #ei = ExpectedImprovement(model=gp, best_f=f_train_rs.max())
    ucb =UpperConfidenceBound(gp, beta=1)

    candidate, acq_value = optimize_acqf(ucb, bounds=bounds, q=1, num_restarts=num_restarts, raw_samples=raw_samples)

    x_rs_next=candidate.detach().numpy()[0]
    #print(x_rs_next)
    #apply inverse scaling transform
    if dim_x==1:
        x_next=scaler_x1.inverse_transform(x_rs_next.reshape(-1, 1)).flatten()
    else:
        x_next=scaler_x1.inverse_transform(x_rs_next)
    #x_next=x_rs_next
    print(x_rs_next,x_next)
    gpr=None
    
    return x_next, x_next, gpr


#for GP parameter values need to be in other range, for instance [0,5], apply scaler to parameter lists
def rescaled_params_df(df):
    scalers = []
    scaled_values = []

    for values in df['Param values']:
        scaler = MinMaxScaler(feature_range=(0, 5))
        scaled = scaler.fit_transform(np.asarray(values*10**20).reshape(-1, 1)).flatten()
        scalers.append(scaler)
        scaled_values.append(scaled)

    df['Scalers'] = scalers
    df['Rs param values'] = scaled_values
    
    return df

#generate linearly scaled rs values, independent of actual parameter scaling
def rescaled_params_2_df(df):
    scalers = []
    scaled_values = []

    for values in df['Param values']:
        scaler = MinMaxScaler(feature_range=(0, 5))
        values_lin=np.linspace(0,5,len(values))
        scaled = scaler.fit_transform(values_lin.reshape(-1, 1)).flatten()
        scalers.append(scaler)
        scaled_values.append(scaled)

    df['Scalers'] = scalers
    df['Rs param values'] = scaled_values
    
    return df

#place holder rescaling
def rescaled_params_no_sc_df(df):
    scalers = []
    scaled_values = []

    for values in df['Param values']:
        scaler = FunctionTransformer(func=None) #MinMaxScaler(feature_range=(0, 5))
        scaled = scaler.fit_transform(np.asarray(values).reshape(-1, 1)).flatten()
        scalers.append(scaler)
        scaled_values.append(scaled)

    df['Scalers'] = scalers
    df['Rs param values'] = scaled_values
    
    return df


def scale_by_inverse_min(X):
    min_vals = np.min(X, axis=0)
    return X * (1 / min_vals)

#sklearn scaler struggles with small values, use simple implementaion generated with Copilot
class MinMaxScaler2:
    def __init__(self):
        self.data_min = None
        self.data_max = None

    def fit(self, data):
        """
        Compute the minimum and maximum values for scaling.

        Parameters:
        data (numpy.ndarray): The input data to be scaled.
        """
        self.data_min = np.min(data, axis=0)
        self.data_max = np.max(data, axis=0)

    def transform(self, data):
        """
        Scale the data to the range [0, 1].

        Parameters:
        data (numpy.ndarray): The input data to be scaled.

        Returns:
        numpy.ndarray: The scaled data.
        """
        if self.data_min is None or self.data_max is None:
            raise ValueError("The scaler has not been fitted yet.")
        scaled_data = (data - self.data_min) / (self.data_max - self.data_min)
        return scaled_data

    def inverse_transform(self, scaled_data):
        """
        Reverse the scaling of the data.

        Parameters:
        scaled_data (numpy.ndarray): The scaled data to be reverted.

        Returns:
        numpy.ndarray: The original data.
        """
        if self.data_min is None or self.data_max is None:
            raise ValueError("The scaler has not been fitted yet.")
        original_data = scaled_data * (self.data_max - self.data_min) + self.data_min
        return original_data
        
#apply MinMaxScaler2 to data
def rescaled_params_df_MinMaxScaler2(df):
    scalers = []
    scaled_values = []

    for values in df['Param values']:
        scaler = MinMaxScaler2()
        scaler.fit(np.asarray(values).reshape(-1, 1))
        scaled = scaler.transform(np.asarray(values).reshape(-1, 1)).flatten()
        scalers.append(scaler)
        scaled_values.append(scaled)

    df['Scalers'] = scalers
    df['Rs param values'] = scaled_values
    
    return df

## Set up optimisation loop

In [ ]:
#load target angle distribution
values_angle_arm_axis_target=np.load("data/values_angle_arm_axis_target.npy")
values_angle_arm_axis_target=values_angle_arm_axis_target.flatten()
arm_axis_angle_distr_target=np.histogram(values_angle_arm_axis_target*180/np.pi,bins=np.arange(0,190,1),density=True)[0]

arm_axis_angle_distr_target=np.asarray(arm_axis_angle_distr_target)


plt.figure(figsize=(6,6))

plt.hist(values_angle_arm_axis_target*180/np.pi,density=True,label="Target arm axis angle",histtype="step")


plt.legend()
plt.xlabel("Angle [deg]",fontname = "Arial",fontsize=11)
plt.ylabel("Normalized frequency",fontname = "Arial",fontsize=11)
plt.xticks(fontname = "Arial",fontsize=11)
plt.yticks(fontname = "Arial",fontsize=11)
plt.grid()
plt.tight_layout()

In [ ]:
#copy of sim params
run_sim_1_input_1=run_sim_input.copy()

#specify which parameters are to be optimised over which range
var_params_1={"Param names":["k_angle_ds_arm"],
                         
            "Param values":[k_angle_ds_arm_range],
            
            "Set of indep params":[0],
           "Index init guess":[[0,3]] #[1,8]
           }

df_var_params_1=pd.DataFrame(var_params_1)



#placeholder scaler on real values
df_var_params_1 = rescaled_params_no_sc_df(df_var_params_1)


df_var_params_1


#specify evaluation and Bayesian opt parameters for each independent set of parameters
#every independent parameter set needs own dictionaries for GP and eval, as parameters need to be updated independently later


GP_params_1={"x_rs":[[0]],"x_train_rs":[[0]],"x":[[0]],"f_train_rs":[[0]],"seed":1,"dim_x":1,"dim_y":1,"num_restarts":100,"raw_samples":1000,"scaler_x":MinMaxScaler2}


Eval_params_1={"folder":None,"names_sim_repeat":None,"t_skip":300,"t_lim":None,"t_step":1,
             "arm_axis_angle_distr_target":arm_axis_angle_distr_target}

eval_opt_param_set_1={"Set of indep params":[0],
                      "Eval fct": [eval_sim_c1_1_aa],
                      "Eval params":[Eval_params_1],
                      "Scaler": [FunctionTransformer(func=None)],
                      "GP": [apply_gp_bo_1],
                      "GP params": [GP_params_1]}

df_eval_opt_param_set_1=pd.DataFrame(eval_opt_param_set_1)
df_eval_opt_param_set_1

In [ ]:
%%time
#run optimization
df_t1,gpr_1=run_gp_11(df_run_sim=run_sim_1_input_1,
              df_var_params=df_var_params_1,
              fct_sim=run_sim_3,
              df_eval_opt_param_set=df_eval_opt_param_set_1,
              sim_folder="sims/aa_opt/",
              sim_name="sim_aa_opt_1_",
              file_name_sim_params="params_aa_opt_1", #"params_aa_opt_1", 
              sim_repeats=5,
              iterations=4,
              print_sim_config=False,
              load_existing_init_sims=True, # use this option to load in existing initial simulations
              load_existing_opt_sims=False) #use this option to loed in exisiting optimisation simulations

In [ ]:
df_t1

In [ ]:
#create dictionary with updated parameters
dict_opt_params_step_1={}
dict_opt_rs_params_step_1={}

for index, row in df_t1.iterrows():
    id_max=np.argmax(row["f train"])
    print("max fitness:",id_max,np.max(row["f train"]))
    opt_params_names=row["Param names in set"]
    opt_params_names_rs=[ e+"_rs" for e in opt_params_names]
    opt_params=row["x train"][id_max]
    opt_params_rs=row["x train rs"][id_max]
    
    for key, value in zip(opt_params_names,opt_params):
        dict_opt_params_step_1[key] = value
    for key, value in zip(opt_params_names_rs,opt_params_rs):
        dict_opt_rs_params_step_1[key] = value    
        
    print("param set:",row["Set of indep params"],row["x train rs"][id_max])
    print("param set:",row["Set of indep params"],row["x train"][id_max])

    
print(dict_opt_rs_params_step_1)    
print(dict_opt_params_step_1)

In [ ]:
#plot the fitness as function of iterations
for index, row in df_t1.iterrows():
    fig1=plt.figure(figsize=(6,3))
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)
    #ax1 = fig1.add_subplot(111)
    fitness_range_l=0.4
    fitness_range_h=1
    fitness_range_step=4
    iterations_step=5
    init_sim_range=2
    
    param_set=row["Set of indep params"]
    param_set_name="Parameter set "+str(param_set)
    print(param_set_name)
    iterations_l=np.arange(1,len(row["f train"])+1)
    param_value_range=(0.0,0.2,0.4,0.6,0.8,1.0) #(10**-22,2.5*10**-19,5*10**-19,7.5*10**-19,10**-18)  #=np.linspace(10**-22, 10**-18,5)  #np.arange(0,6,1) 
    fitnes_range=np.linspace(fitness_range_l,fitness_range_h,fitness_range_step)
    ax1.errorbar(iterations_l,row["f train"],color="black")
    id_max_fitnes=np.argmax(row["f train"])
    
    ax1.axvspan(1, init_sim_range, facecolor='grey', alpha=0.5,label="Initial simulations")
    ax1.axvline(iterations_l[id_max_fitnes],color="red",label="Best fitness")
    ax1.set_ylabel("Fitness",fontname = "Arial",fontsize=11)
    #ax1.set_xlabel("Iterations",fontname = "Arial",fontsize=11)
    ax1.set_xticks(iterations_l[::iterations_step],iterations_l[::iterations_step],fontname = "Arial",fontsize=11)
    ax1.set_yticks(fitnes_range,fitnes_range ,fontname= "Arial",fontsize=11)
    ax1.legend(loc="lower left",prop={'family': 'Arial','size': 11})
    ax1.grid()
    
    
    #for j in range(len(row["x train"][0])):
      #  param_name=row["Param names in set"][j]
       # ax2.errorbar(iterations_l,[sublist[j] for sublist in row["x train"]],label=param_name)
    for j in range(len(row["x train rs"][0])):
        param_name=row["Param names in set"][j]
        param_sel=[sublist[j] for sublist in row["x train rs"]]
        scaler_param_sel=MinMaxScaler2()
        param_sel_range=df_var_params_1[df_var_params_1["Param names"]==param_name]["Param values"].to_numpy()[0]
        scaler_param_sel.fit(param_sel_range)
        param_sel_rs=scaler_param_sel.transform(param_sel)
        ax2.errorbar(iterations_l,param_sel_rs,label=param_name)
        
    ax2.axvspan(1, init_sim_range, facecolor='grey', alpha=0.5,label="Initial simulations")
    ax2.axvline(iterations_l[id_max_fitnes],color="red",label="Best fitness")
    ax2.set_xlabel("Iterations",fontname = "Arial",fontsize=11)
    ax2.set_ylabel("Rs. param. values",fontname = "Arial",fontsize=11)
    ax2.set_yticks(param_value_range,param_value_range,fontname = "Arial",fontsize=11)
    #ax2.yaxis.set_major_formatter(matplotlib.ticker.FormatStrFormatter('%.1f'))
    ax2.legend(loc="lower left",prop={'family': 'Arial','size': 11})
    ax2.grid()
    plt.show()

Details on the other angles, visualisation etc. can be found in `NCG_BO_opt_x_motif.ipynb`

## 5. Setting up a hydrogel simulation (many nanostars, equilibration + production)

Once all bead-spring parameters are optimised, we simulate networks of 100 nanostars in a periodic box
matching the experimental nanostar concentration. Each such simulation is run in two stages:

1. **Equilibration** — nanostars are placed randomly and allowed to relax sterically, with sticky-end
   reactions disabled (`k_b_x_c1 = k_ub_x_c1 = 0`) in this example here, and the final configuration saved as a checkpoint.
2. **Production** — starting from the equilibration checkpoint, sticky-end fusion/splitting reactions are
   enabled and the network is allowed to assemble over a long run, from which network structure and
   rheological properties are later extracted.



In [ ]:
box_size = [69.718, 69.718, 69.718]  # matches experimental nanostar concentration

run_sim_hydrogel = run_sim_input.copy()
run_sim_hydrogel.update({
    "box_size": box_size,
    "periodic_bc_x": True, "periodic_bc_y": True, "periodic_bc_z": True,
    "num_x_motif_c1": 100,
    "x_motif_c1_ini": [np.asarray([e]) for e in
                        np.random.uniform(size=(100, 3)) * np.asarray(box_size) - np.asarray(box_size) * 0.5],
    "x_motif_c1_rot": True, #motifs are placed in random orientation
    "rec_stride": 5000,
})



In [ ]:
import os #to create folders

In [ ]:
%%capture
#create equilibration checkpoints


#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(5):
    folder_ch="sims/hydrogel/eq_checkpoint_"+str(i)+"/"
    folder_pr="sims/hydrogel/prod_checkpoint_"+str(i)+"/"
    os.makedirs(folder_ch)
    os.makedirs(folder_pr)
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_input.copy()
    run_sim_1_input_c["t_tot"]=2*10**6
    run_sim_1_input_c["k_b_x_c1"]=0    
    run_sim_1_input_c["k_ub_x_c1"]=0  
    run_sim_1_input_c["k_b_x_hp_c1"]=0
    run_sim_1_input_c["k_ub_x_hp_c1"]=0
    run_sim_1_input_c["new_checkpoint"]=folder_ch
    #run_sim_1_input_c["checkpoint_path"]= "/home/aaron/phd/Nanomotif_Coarse_Graining_1/ML_parameter_opt_1/NCG_X_motif_conrad_1/shear_mod_test_1/checkpoints_15/"
 
    run_sim_1_input_c["folder"]=folder_ch
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_eq_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)


In [ ]:
%%capture
#Production runs
#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(5):
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_input.copy()
    run_sim_1_input_c["t_tot"]=int(1.5*10**7)
    run_sim_1_input_c["k_ub_x_c1"]=0
    run_sim_1_input_c["new_checkpoint"]="sims/hydrogel/prod_checkpoint_"+str(i)+"/"
    run_sim_1_input_c["checkpoint_path"]="sims/hydrogel/eq_checkpoint_"+str(i)+"/"
 
    run_sim_1_input_c["folder"]="sims/hydrogel/prod_checkpoint_"+str(i)+"/"
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_prod_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)

## 6. Evaluating properties of ReaDDy hydrogel simulations

From the production trajectories we extract: potential energy and the resulting storage/loss moduli G', G'', with the option to average over the five
independent repeats. More details can be found in `NCG_network_properties_3`.

In [ ]:
import readdy
from NCG_readdy_sim_eval_5 import *
#load saved config
traj_mech=readdy.Trajectory("sims/hydrogel/x_c_mech_opt_prod_0.h5")

times_mech,types_mech,ids_mech,part_positions_mech=traj_mech.read_observable_particles()
time_pt_mech, counts_pt_mech = traj_mech.read_observable_number_of_particles()
#part_pos_eq=part_positions

time_e_mech, energy_mech=traj_mech.read_observable_energy()

time_obs_top_mech,obs_top_mech=traj_mech.read_observable_topologies()

In [ ]:
fig=plt.figure(figsize=(9/2.54,3.5/2.54))

plt.errorbar(time_e_mech, energy_mech,color="black")


plt.xticks(fontname = "Arial",fontsize=11)
plt.yticks([1.8*10**-17,2.3*10**-17],fontname = "Arial",fontsize=11)
plt.ylabel("Energy [ReaDDy]",fontname = "Arial",fontsize=11)
plt.xlabel("Time [ReaDDy]",fontname = "Arial",fontsize=11)
plt.grid()


In [ ]:
#to evaluate a larger batch, many folders can be specified here, 

folder_name_ex1=["sims/hydrogel/"+str(e)+"/x_c_mech_opt_prod_"+str(e)+".h5" for e in [0,1,2,3,4]] #set the number of repeats to average



#gather all simulation names in one list
folder_names_all_ex=[folder_name_ex1]

In [ ]:
%%time
#calculate G' and G'' from simulations using 

omega_t_sim1_hydr=np.geomspace(10**-3,10**5,10**3) #geomspace(10**2,10**10,10**3)
t_sim1_hydr=np.geomspace(10**-2,10**3,10**2)


folder_names_dyn_mod=[folder_names_all_ex[e] for e in [0]] #folders to read in 
time_skip_dyn_mod=[2250] #first time steps to skeip

input_dyn_mod=[]
box_size=[69.7,69.7,69.7]
eta=8*10**-3
a=4.7
t_r_avg_f=-10
t_r_avg_l=10

from concurrent.futures import ThreadPoolExecutor

for i in range(len(folder_names_dyn_mod)):
    input_dyn_mod.append((folder_names_dyn_mod[i],omega_t_sim1_hydr,t_sim1_hydr,1,1,1,1,1,15,time_skip_dyn_mod[i],box_size,eta,a,t_r_avg_f,t_r_avg_l))

with ThreadPoolExecutor(max_workers=18) as executor:
    results_dyn_mod_hydr = list(executor.map(lambda args: avg_moduli_hydr_3(*args), input_dyn_mod))



In [ ]:
# split results in storage, loss and cross links
storage_mod_all_sims_all_hydr=[results_dyn_mod_hydr[e][0] for e in range(len(results_dyn_mod_hydr))]
loss_mod_all_sims_all_hydr=[results_dyn_mod_hydr[e][1] for e in range(len(results_dyn_mod_hydr))]
cross_links_all_sims_all_hydr=[results_dyn_mod_hydr[e][2] for e in range(len(results_dyn_mod_hydr))]

In [ ]:


#fig=plt.figure(figsize=(3.5/2.54,3.5/2.54))
#ax1 = fig.add_subplot(111)
for i in range(len(storage_mod_all_sims_all_hydr)):
    fig=plt.figure(figsize=(3.5/2.54,3.5/2.54))
    ax1 = fig.add_subplot(111)
    
    mean_storage_mod_all_sims_ex1=np.mean(storage_mod_all_sims_all_hydr[i],axis=0)
    std_e_storage_mod_all_sims_ex1=np.std(storage_mod_all_sims_all_hydr[i],axis=0)/(np.sqrt(len(storage_mod_all_sims_all_hydr[i])))
    
    
    mean_loss_mod_all_sims_ex1=np.mean(loss_mod_all_sims_all_hydr[i],axis=0)
    std_e_loss_mod_all_sims_ex1=np.std(loss_mod_all_sims_all_hydr[i],axis=0)/(np.sqrt(len(loss_mod_all_sims_all_hydr[i])))
    
    ax1.errorbar(omega_t_sim1_hydr[::],mean_storage_mod_all_sims_ex1[::],fmt="-",label=r"$G'$",color="black",linewidth=0.7)
    ax1.fill_between(omega_t_sim1_hydr[::], mean_storage_mod_all_sims_ex1[::] + std_e_storage_mod_all_sims_ex1[::], mean_storage_mod_all_sims_ex1[::] - std_e_storage_mod_all_sims_ex1[::], alpha = 0.3 ,facecolor="black")
    
    
    ax1.errorbar(omega_t_sim1_hydr[::],mean_loss_mod_all_sims_ex1[::],fmt="-",label=r"$G''-\omega \eta$",color="red",linewidth=0.7)
    ax1.fill_between(omega_t_sim1_hydr[::], mean_loss_mod_all_sims_ex1[::] + std_e_loss_mod_all_sims_ex1[::], mean_loss_mod_all_sims_ex1[::] - std_e_loss_mod_all_sims_ex1[::], alpha = 0.3 ,facecolor="red")

    #determine scaling
    n1=np.gradient(mean_storage_mod_all_sims_ex1,omega_t_sim1_hydr)/mean_storage_mod_all_sims_ex1 *omega_t_sim1_hydr
    n2=np.gradient(mean_loss_mod_all_sims_ex1,omega_t_sim1_hydr)/mean_loss_mod_all_sims_ex1 *omega_t_sim1_hydr
    # Condition: 0 < n < 1 for both
    mask = (n1 > 0) & (n1 < 1) & (n2 > 0) & (n2 < 1)
    
    # Find indices where condition holds
    indices = np.where(mask)[0]
    
    start_idx = indices[0]
    end_idx = indices[-1] 

    ax1.axvspan(omega_t_sim1_hydr[start_idx], omega_t_sim1_hydr[end_idx], color='gray', alpha=0.2)

    
    omega_inters_target_1,dyn_mod_inters_target_1=find_intersection(x1=omega_t_sim1_hydr, y1=mean_storage_mod_all_sims_ex1, x2=omega_t_sim1_hydr, y2=mean_loss_mod_all_sims_ex1)[0]
    print("Intersection:","omega:",omega_inters_target_1,"dynamic mod.", dyn_mod_inters_target_1)
   
    
    #ax1.errorbar(omega_t_sim1[:200],omega_t_sim1[:200]**2, label=r"$\omega^2$")
    #ax1.errorbar(omega_t_sim1[:200],omega_t_sim1[:200]**1 , label=r"$\omega$")
    #ax1.errorbar(omega_t_sim1[:200],omega_t_sim1[:200]**0.5 , label=r"$\omega^{0.5}$")
    #ax1.errorbar(omega_t_sim1[700:],omega_t_sim1[700:]**-1 , label=r"$\omega^{-1}$")
    fs1=8
    plt.xscale("log")
    plt.yscale("log")
    plt.grid()
    #plt.ylim(10**-6,2)
    #plt.xlim(10**-3,10**3)
    plt.xticks([10**-2,10**1,10**4],fontname = "Arial",fontsize=fs1)
    plt.yticks([10**-6,10**-3,10**0],fontname = "Arial",fontsize=fs1)
    #plt.xticks([10**-2,10**0,10**2],fontname = "Arial",fontsize=fs1)
    #plt.yticks([10**-6,10**-4,10**-2,10**0],fontname = "Arial",fontsize=fs1)
    ax1.set_ylabel("Dynamic modulus [a.u.]",fontname = "Arial",fontsize=fs1)
    ax1.set_xlabel("Frequency [a.u.]",fontname = "Arial",fontsize=fs1)
    #ax1.legend(prop={'family': 'Arial','size': fs1},loc="lower right")
    #if i ==0:
    save_name_1="dyn_mod_ub_rate_hydr_"+str(i)+"_1b.pdf"
    #plt.savefig(save_name_1,format="pdf",bbox_inches="tight")
    plt.show()

## 7. Generating theoretical graphs and their G', G'' curves

Beyond simulated networks, we can generate purely theoretical random graphs with a specified number of
nodes, edge count, and maximum node degree (valence), and evaluate their predicted G'/G'' curves with the
same graph-Laplacian approach. This is useful to isolate the effect of connectivity/valence from other
molecular details. `rd_inters_max4` generates a
batch of random graphs for given parameters and returns their storage/loss moduli directly.

In [ ]:
from NCG_theo_graphs_1 import rd_inters_max4

omega_t_theo = np.geomspace(1e-3, 1e5, 1000)
t_theo = np.geomspace(1e-2, 1e3, 100)

# A few theoretical graphs of 100 nodes, maximum node degree 4 (matching four-armed nanostars),
# with edge counts spanning from near-percolation to fully connected.
(f_intersection, f_max_loss_mod, len_f_range, storage_mod_theo, loss_mod_theo,
 rd_elem, relax_times) = rd_inters_max4(
    num_samples=5, nodes=100, n_elem_max=200, n_elem_min=100, max_edges_per_node=4,
    omega_test=omega_t_theo, t_test=t_theo,
)

plt.figure(figsize=(4, 3))
for i in range(len(storage_mod_theo)):
    plt.plot(omega_t_theo, storage_mod_theo[i], label=f"edges={rd_elem[i]}")
plt.xscale("log")
plt.yscale("log")
plt.xlabel(r"$\omega$ [a.u.]")
plt.ylabel(r"$G'$ [a.u.]")
plt.legend(fontsize=7)
plt.grid()
plt.show()


## Beyond this notebook
The remaining parts of the workflow are available in the full reference notebooks in this repository:

- **Thermodynamics of sticky ends from oxDNA**  — using oxDNA to compute free energies of hybridisation between sticky ends and haripin formation withtin (`NCG_thermodyn_x_motif`).
- **Sticky-end sequence design with NUPACK and Bayesian optimisation** — predicting DNA sequences,
  temperature, and salt conditions that yield a target fraction of hybridised sticky ends
  (`NCG_SE_seq_prediction`).
- **Inverse design** — combining predicted network connectivity (from simulated or theoretical graphs, as
  in Sections 6-7 above) with the sequence-design step above to go from a target G', G'' response to
  concrete sticky-end sequences and conditions 
  `NCG_ML_rheo_pred`.
- **Additional figure, robustness and validation analyses** 
  (`NCG_network_properties_3`).
- **The full set of six angle distributions** (base, opposing-arm, sticky-end, hairpin sticky-end,
  arm-axis, and fused-arm angles) and the complete, split Bayesian optimisation strategy across all
  bead-spring model parameters (`NCG_target_angles_x_motif`, `NCG_BO_opt_x_motif`).
- **Production-scale hydrogel simulations** with checkpointing across multiple simulation stages and
  conditions (`NCG_run_networking_sim_1`).